<a href="https://colab.research.google.com/github/jeetbhatnagar89-beep/Phi-4-training/blob/develop/phi_4_conversational.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [ ]:
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [1]:
from unsloth import FastLanguageModel  # FastVisionModel for LLMs
import torch
max_seq_length = 2048  # Choose any! We auto support RoPE Scaling internally!
load_in_4bit = True  # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit",  # Llama-3.1 2x faster
    "unsloth/Mistral-Small-Instruct-2409",  # Mistral 22b 2x faster!
    "unsloth/Phi-4",  # Phi-4 2x faster!
    "unsloth/Phi-4-unsloth-bnb-4bit",  # Phi-4 Unsloth Dynamic 4-bit Quant
    "unsloth/gemma-2-9b-bnb-4bit",  # Gemma 2x faster!
    "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",  # Qwen 2.5 2x faster!
    "unsloth/Llama-3.2-1B-bnb-4bit",  # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
]  # More models at https://unsloth.ai/docs/get-started/all-our-models

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-4",
    max_seq_length = 2048,
    load_in_4bit = True,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.8: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.39G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.5.8 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


<a name="Data"></a>
### Data Prep
We now use the `Phi-4` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. But we convert it to HuggingFace's normal multiturn format `("role", "content")` instead of `("from", "value")`/ Phi-4 renders multi turn conversations like below:

```
<|im_start|>user<|im_sep|>Hello!<|im_end|>
<|im_start|>assistant<|im_sep|>Hi! How can I help?<|im_end|>
<|im_start|>user<|im_sep|>What is 2+2?<|im_end|>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, phi4, llama3` and more.

### Custom Dataset for HTML Buttons

I'll create a custom dataset containing prompts for generating HTML code for Primary, Secondary, and Tertiary buttons, based on your specifications. This dataset will replace the default `mlabonne/FineTome-100k` dataset for training.

In [5]:
import re
import random
from datasets import Dataset

def standardize_background_color(html_string, button_type):
    target_color = {
        'primary': 'red',
        'secondary': 'grey',
        'tertiary': 'black'
    }.get(button_type)

    if target_color:
        # Regex to find and replace background-color in style attribute
        # This handles cases where background-color might not be the first or last style property
        updated_html = re.sub(
            r'(background-color:)[^;]+;',
            f'background-color: {target_color};',
            html_string
        )
        return updated_html
    return html_string

# Define the custom dataset in ShareGPT style
custom_data = [
    {
        "conversations": [
            {"from": "user", "value": "Give me primary button"},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Primary Button</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Create a secondary button"},
            {"from": "gpt", "value": '<button style="background-color: grey; border-radius: 5px; color: black; padding: 20px;">Secondary Button</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Show me a tertiary button"},
            {"from": "gpt", "value": '<button style="background-color: black; border-radius: 5px; color: white; padding: 20px;">Tertiary Button</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Generate HTML for a primary button."},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Click Me</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "I need a secondary button HTML code."},
            {"from": "gpt", "value": '<button style="background-color: grey; border-radius: 5px; color: black; padding: 20px;">Submit</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Provide tertiary button HTML."},
            {"from": "gpt", "value": '<button style="background-color: black; border-radius: 5px; color: white; padding: 20px;">Learn More</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Red button, white text, rounded, 20px padding."},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Custom Button</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Grey background, black text, rounded, 20px padding button."},
            {"from": "gpt", "value": '<button style="background-color: grey; border-radius: 5px; color: black; padding: 20px;">Go</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Black button, white text, rounded, 20px padding."},
            {"from": "gpt", "value": '<button style="background-color: black; border-radius: 5px; color: white; padding: 20px;">Dismiss</button>'} # 9 examples
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Can you generate a primary action button?"},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Action</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "I need a secondary button for my web page."},
            {"from": "gpt", "value": '<button style="background-color: grey; border-radius: 5px; color: black; padding: 20px;">More Info</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Produce a tertiary style button in HTML."},
            {"from": "gpt", "value": '<button style="background-color: black; border-radius: 5px; color: white; padding: 20px;">Cancel</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "HTML for a primary button with 'Save' text."},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Save</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Generate a secondary button with 'Edit' text."},
            {"from": "gpt", "value": '<button style="background-color: grey; border-radius: 5px; color: black; padding: 20px;">Edit</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "I want a tertiary button that says 'Delete'."},
            {"from": "gpt", "value": '<button style="background-color: black; border-radius: 5px; color: white; padding: 20px;">Delete</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Create a button with red background, white text, rounded corners, and 20px padding."},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Confirm</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "A button that is grey, black text, rounded, and 20px padded."},
            {"from": "gpt", "value": '<button style="background-color: grey; border-radius: 5px; color: black; padding: 20px;">Upload</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Give me a black button with white text, rounded edges, and 20px padding."},
            {"from": "gpt", "value": '<button style="background-color: black; border-radius: 5px; color: white; padding: 20px;">Close</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Show me an HTML primary button example."},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Click Here</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "I need a button that is red with white text, round edges, and 20px padding."},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Shop Now</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Generate a grey button with black text, rounded corners and 20px padding."},
            {"from": "gpt", "value": '<button style="background-color: grey; border-radius: 5px; color: black; padding: 20px;">View Details</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "I want a black button with white text, rounded corners and 20px padding."},
            {"from": "gpt", "value": '<button style="background-color: black; border-radius: 5px; color: white; padding: 20px;">Download</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Provide HTML for a primary button with 'Start' text."},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Start</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Give me a secondary button with 'Reset' text."},
            {"from": "gpt", "value": '<button style="background-color: grey; border-radius: 5px; color: black; padding: 20px;">Reset</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Create a tertiary button with 'Info' text."},
            {"from": "gpt", "value": '<button style="background-color: black; border-radius: 5px; color: white; padding: 20px;">Info</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Can you make an HTML button that is red with white text and has 20px padding?"},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Proceed</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "I need an HTML button that is grey with black text and 20px padding."},
            {"from": "gpt", "value": '<button style="background-color: grey; border-radius: 5px; color: black; padding: 20px;">Back</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Generate an HTML button with a black background, white text, and 20px padding."},
            {"from": "gpt", "value": '<button style="background-color: black; border-radius: 5px; color: white; padding: 20px;">Continue</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "I want the HTML for a primary button, round edges."},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Login</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Show me a secondary button with a grey background and black text."},
            {"from": "gpt", "value": '<button style="background-color: grey; border-radius: 5px; color: black; padding: 20px;">Explore</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "I need a tertiary button with a black background, white text, and rounded corners."},
            {"from": "gpt", "value": '<button style="background-color: black; border-radius: 5px; color: white; padding: 20px;">Discard</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Generate a red button with white text and 20px padding."},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Next</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Give me a button with a grey background, black text, rounded and 20px padding."},
            {"from": "gpt", "value": '<button style="background-color: grey; border-radius: 5px; color: black; padding: 20px;">Previous</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Provide HTML for a black button with white text, rounded edges, and 20px padding."},
            {"from": "gpt", "value": '<button style="background-color: black; border-radius: 5px; color: white; padding: 20px;">Exit</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Can you make a primary button that says 'Register'?"},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Register</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "I want an HTML secondary button with the label 'Settings'."},
            {"from": "gpt", "value": '<button style="background-color: grey; border-radius: 5px; color: black; padding: 20px;">Settings</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Generate a tertiary button for 'Help'."},
            {"from": "gpt", "value": '<button style="background-color: black; border-radius: 5px; color: white; padding: 20px;">Help</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Red, white text, 20px padding, round button."},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Submit Order</button>'} # 39 examples
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "HTML for a primary button with no specific text, default styles."},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Button</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Secondary button with a blue background and white text."},
            {"from": "gpt", "value": standardize_background_color('<button style="background-color: blue; border-radius: 5px; color: white; padding: 20px;">Info</button>', 'secondary')} # Modified for consistency
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Tertiary button, green background, black text, no border radius."},
            {"from": "gpt", "value": standardize_background_color('<button style="background-color: green; border-radius: 0px; color: black; padding: 20px;">Custom</button>', 'tertiary')} # Modified for consistency
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Primary button with 'Proceed' text and a 10px border radius."},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 10px; color: white; padding: 20px;">Proceed</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Secondary button with 'Cancel' text, grey background, blue text."},
            {"from": "gpt", "value": '<button style="background-color: grey; border-radius: 5px; color: blue; padding: 20px;">Cancel</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Tertiary button with 'Discard' text, yellow background, black text, 15px border radius."},
            {"from": "gpt", "value": standardize_background_color('<button style="background-color: yellow; border-radius: 15px; color: black; padding: 20px;">Discard</button>', 'tertiary')} # Modified for consistency
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "HTML for a primary button with a large padding of 30px."},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 5px; color: white; padding: 30px;">Large Button</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Secondary button, small padding of 10px, grey background, white text."},
            {"from": "gpt", "value": '<button style="background-color: grey; border-radius: 5px; color: white; padding: 10px;">Small Button</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Tertiary button with 'Finish' text, orange background, white text, no border radius, 25px padding."},
            {"from": "gpt", "value": standardize_background_color('<button style="background-color: orange; border-radius: 0px; color: white; padding: 25px;">Finish</button>', 'tertiary')} # Modified for consistency
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Primary button, green background, white text, rounded corners."},
            {"from": "gpt", "value": standardize_background_color('<button style="background-color: green; border-radius: 5px; color: white; padding: 20px;">Go</button>', 'primary')} # Modified for consistency
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Secondary button, purple background, yellow text, 10px border radius."},
            {"from": "gpt", "value": standardize_background_color('<button style="background-color: purple; border-radius: 10px; color: yellow; padding: 20px;">Next Step</button>', 'secondary')} # Modified for consistency
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Tertiary button, brown background, light grey text, 0px border radius, 15px padding."},
            {"from": "gpt", "value": standardize_background_color('<button style="background-color: brown; border-radius: 0px; color: lightgrey; padding: 15px;">Archive</button>', 'tertiary')} # Modified for consistency
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Give me a primary and tertiary button."},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Primary</button><button style="background-color: black; border-radius: 5px; color: white; padding: 20px;">Tertiary</button>'}
        ]
    },
    {
        "conversations": [
            {"from": "user", "value": "Show me an HTML primary button with a 'Submit' label and a tertiary button with an 'Abort' label."},
            {"from": "gpt", "value": '<button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Submit</button><button style="background-color: black; border-radius: 5px; color: white; padding: 20px;">Abort</button>'}
        ]
    }
]

def generate_synthetic_examples(num_examples=10):
    new_examples = []
    button_texts = ["Confirm", "Decline", "Accept", "Reject", "Proceed", "Go Back", "Save Draft", "Publish"]
    border_radii = [0, 5, 10, 15, 20]
    paddings = [10, 15, 20, 25, 30]
    colors = {"primary": "red", "secondary": "grey", "tertiary": "black"}
    text_colors = ["white", "black", "blue", "green"]

    for _ in range(num_examples):
        button_type = random.choice(["primary", "secondary", "tertiary"])
        text = random.choice(button_texts)
        border_radius = random.choice(border_radii)
        padding = random.choice(paddings)
        bg_color = colors[button_type]
        color = random.choice(text_colors)

        user_prompt = f"Generate an HTML {button_type} button with '{text}' text, {border_radius}px border radius, {padding}px padding, and {color} text."
        gpt_response = f'<button style="background-color: {bg_color}; border-radius: {border_radius}px; color: {color}; padding: {padding}px;">{text}</button>'
        new_examples.append({"conversations": [{"from": "user", "value": user_prompt}, {"from": "gpt", "value": gpt_response}]})

        # Add combination examples
        if random.random() < 0.5: # 50% chance to add a combination
            second_button_type = random.choice([bt for bt in ["primary", "secondary", "tertiary"] if bt != button_type])
            second_text = random.choice(button_texts)
            second_border_radius = random.choice(border_radii)
            second_padding = random.choice(paddings)
            second_bg_color = colors[second_button_type]
            second_color = random.choice(text_colors)

            user_prompt_combo = f"Give me an HTML {button_type} button with '{text}' and a {second_button_type} button with '{second_text}'."
            gpt_response_combo = f'<button style="background-color: {bg_color}; border-radius: {border_radius}px; color: {color}; padding: {padding}px;">{text}</button><button style="background-color: {second_bg_color}; border-radius: {second_border_radius}px; color: {second_color}; padding: {second_padding}px;">{second_text}</button>'
            new_examples.append({"conversations": [{"from": "user", "value": user_prompt_combo}, {"from": "gpt", "value": gpt_response_combo}]})

    return new_examples

synthetic_data = generate_synthetic_examples(num_examples=30) # Generate 30 more synthetic examples
custom_data.extend(synthetic_data)

dataset = Dataset.from_list(custom_data)

print(f"Custom dataset created with {len(dataset)} examples.")

Custom dataset created with 99 examples.


In [6]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "phi-4",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize = False, add_generation_prompt = False
        )
        for convo in convos
    ]
    return { "text" : texts, }

# The dataset variable already contains our custom data, no need to load from HuggingFace.
# from datasets import load_dataset
# dataset = load_dataset("mlabonne/FineTome-100k", split = "train")

We now use `standardize_sharegpt` to convert ShareGPT style datasets into HuggingFace's generic format. This changes the dataset from looking like:
```
{"from": "system", "value": "You are an assistant"}
{"from": "human", "value": "What is 2+2?"}
{"from": "gpt", "value": "It's 4."}
```
to
```
{"role": "system", "content": "You are an assistant"}
{"role": "user", "content": "What is 2+2?"}
{"role": "assistant", "content": "It's 4."}
```

In [7]:
from unsloth.chat_templates import standardize_sharegpt

dataset = standardize_sharegpt(dataset)
dataset = dataset.map(
    formatting_prompts_func,
    batched = True,
)

Unsloth: Standardizing formats (num_proc=6):   0%|          | 0/99 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

We look at how the conversations are structured for item 5:

In [8]:
dataset[5]["conversations"]

[{'role': 'user', 'content': 'Provide tertiary button HTML.'},
 {'role': 'assistant',
  'content': '<button style="background-color: black; border-radius: 5px; color: white; padding: 20px;">Learn More</button>'}]

And we see how the chat template transformed these conversations.

In [9]:
dataset[5]["text"]

'<|im_start|>user<|im_sep|>Provide tertiary button HTML.<|im_end|><|im_start|>assistant<|im_sep|><button style="background-color: black; border-radius: 5px; color: white; padding: 20px;">Learn More</button><|im_end|>'

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!!

In [10]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/99 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs.

In [11]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user<|im_sep|>",
    response_part = "<|im_start|>assistant<|im_sep|>",
)

Map (num_proc=6):   0%|          | 0/99 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/99 [00:00<?, ? examples/s]

We verify masking is actually done:

In [12]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

'<|im_start|>user<|im_sep|>Provide tertiary button HTML.<|im_end|><|im_start|>assistant<|im_sep|><button style="background-color: black; border-radius: 5px; color: white; padding: 20px;">Learn More</button><|im_end|>'

In [13]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'            <button style="background-color: black; border-radius: 5px; color: white; padding: 20px;">Learn More</button><|im_end|>'

We can see the System and Instruction prompts are successfully masked!

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [14]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 99 | Num Epochs = 3 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 65,536,000 of 14,725,043,200 (0.45% trained)


Step,Training Loss
1,2.065900
2,1.899800
3,1.975900
4,1.780200
5,1.101100
6,0.863100
7,0.546000
8,0.390000
9,0.380500
10,0.324300


In [15]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

NameError: name 'start_gpu_memory' is not defined

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!



We use `min_p = 0.1` and `temperature = 1.5`.

In [16]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "phi-4",
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Create HTML code for a primary button"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(
    input_ids = inputs, max_new_tokens = 64, use_cache = True, temperature = 1.5, min_p = 0.1
)
tokenizer.batch_decode(outputs)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


['<|im_start|>user<|im_sep|>Create HTML code for a primary button<|im_end|><|im_start|>assistant<|im_sep|><button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Primary Button</button><|im_end|>']

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [17]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Create HTML code for a primary button"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(
    input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
    use_cache = True, temperature = 1.5, min_p = 0.1
)

<button style="background-color: red; border-radius: 5px; color: white; padding: 20px;">Primary Button</button><|im_end|>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("phi_lora")  # Local saving
tokenizer.save_pretrained("phi_lora")
# model.push_to_hub("your_name/phi_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/phi_lora", token = "YOUR_HF_TOKEN") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "phi_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Describe a tall tower in the capital of France."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(
    input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
    use_cache = True, temperature = 1.5, min_p = 0.1
)

You can also use Hugging Face's `AutoPeftModelForCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer

    model = AutoPeftModelForCausalLM.from_pretrained(
        "phi_lora",  # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("phi_lora")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("phi_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/phi_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("phi_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/phi_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("phi_lora")
    tokenizer.save_pretrained("phi_lora")
if False:
    model.push_to_hub("HF_USERNAME/phi_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/phi_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list in our [Docs](https://unsloth.ai/docs/basics/saving-and-using-models/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("phi_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/phi_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("phi_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/phi_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("phi_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/phi_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/phi_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
    )

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>